[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/05_optimization/04.1_quantization/lab.ipynb)
[![Open In Molab](https://molab.marimo.io/badge.svg)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/05_optimization/04.1_quantization/lab.ipynb)

# Lab 4.1: Quantization

INT4 doubles batch size with <1% quality loss. This lab implements quantization from scratch,
compares AWQ/GPTQ/naive methods, and measures perplexity-speed-memory tradeoffs.

In [ ]:
# Install required packages using subprocess (avoids pip shell issues)
import subprocess
import sys

# Run pip install as a subprocess for reliability
packages = ["torch", "numpy", "matplotlib"]
for pkg in packages:
    # Check if package is already importable before installing
    try:
        __import__(pkg)
    except ImportError:
        # Install missing package via subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Dependencies ready.")

In [ ]:
# === Core imports for quantization experiments ===
import torch          # tensor operations and GPU acceleration
import numpy as np    # numerical utilities for plotting
import matplotlib.pyplot as plt  # visualization of error distributions
import time           # timing quantization overhead
from dataclasses import dataclass  # structured model specs

# Use clean plot style for publication-quality figures
plt.style.use("seaborn-v0_8-darkgrid")

# Detect available hardware (GPU accelerates large matrix ops)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Fix random seed so all experiments are reproducible
torch.manual_seed(42)
# Note: all experiments run on CPU for portability; GPU optional


## 1. Core Quantization Implementations

Symmetric per-tensor and per-group quantization at INT8, INT4, and NF4 precision.

In [ ]:
def quantize_symmetric(tensor: torch.Tensor, bits: int) -> torch.Tensor:
    """Symmetric quantization: map floats to integers using max-abs scaling."""
    # Maximum representable integer for this bit width
    # INT8: 127, INT4: 7, INT2: 1
    qmax = 2 ** (bits - 1) - 1
    # Scale factor maps the full float range to integer range
    # A single outlier inflates this, increasing error for ALL weights
    scale = tensor.abs().max() / qmax
    # Quantize: divide by scale, round to nearest int, clamp to valid range
    quantized = torch.clamp(torch.round(tensor / scale), -qmax - 1, qmax)
    # Dequantize: multiply back by scale to get approximate float values
    return quantized * scale


def quantize_per_group(tensor: torch.Tensor, bits: int, group_size: int = 128) -> torch.Tensor:
    """Per-group quantization: each group of columns gets its own scale."""
    rows, cols = tensor.shape
    # Output tensor to accumulate group-by-group results
    result = torch.zeros_like(tensor)
    # Process columns in groups (typical group_size=128 for AWQ/GPTQ)
    for i in range(0, cols, group_size):
        # Extract one group slice
        group = tensor[:, i:i+group_size]
        # Each group has its own scale factor, isolating outliers
        result[:, i:i+group_size] = quantize_symmetric(group, bits)
    return result


def quantize_nf4(tensor: torch.Tensor) -> torch.Tensor:
    """NF4: information-optimal 4-bit format for normally-distributed weights.
    Used by bitsandbytes for QLoRA fine-tuning."""
    # 16 fixed quantization levels optimized for N(0,1) distribution
    # These minimize expected quantization error for Gaussian-distributed data
    nf4_levels = torch.tensor([
        -1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.0911, 0.0,
        0.0796, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7230, 1.0])
    # Normalize tensor to [-1, 1] range using max absolute value
    abs_max = tensor.abs().max()
    if abs_max == 0:
        return tensor.clone()  # avoid division by zero
    normalized = tensor / abs_max
    # Snap each weight to its nearest NF4 level (argmin of distance)
    indices = (normalized.unsqueeze(-1) - nf4_levels).abs().argmin(dim=-1)
    # Rescale back to original magnitude
    return nf4_levels[indices] * abs_max


def measure_error(original: torch.Tensor, quantized: torch.Tensor) -> dict:
    """Compute quantization error metrics between original and quantized tensors."""
    # Element-wise difference reveals error distribution
    diff = original - quantized
    # Mean squared error (lower = better)
    mse = (diff ** 2).mean().item()
    # Signal power for SQNR calculation
    signal_power = (original ** 2).mean().item()
    # SQNR in dB: signal-to-quantization-noise ratio (higher = better)
    # 30+ dB is generally acceptable for inference
    sqnr = 10 * np.log10(signal_power / max(mse, 1e-10))
    return {"mse": mse, "mae": diff.abs().mean().item(), "sqnr_db": sqnr}


## 2. Method Comparison: Naive vs Per-Group vs AWQ-style

In [ ]:
# Create synthetic LLM weight matrix mimicking 7B-scale hidden dimension
weights = torch.randn(4096, 4096) * 0.02  # std=0.02 typical for LLM layers

# Inject realistic outliers (2% of values are 8x normal magnitude)
# This mimics the outlier distribution observed in real LLMs
outlier_mask = torch.rand_like(weights) < 0.02
weights[outlier_mask] *= 8.0  # outliers destroy naive quantization

# Define all quantization methods to benchmark
methods = {
    # FP16: baseline precision truncation (negligible loss)
    "FP16": lambda t: t.to(torch.float16).to(torch.float32),
    # INT8: 8-bit symmetric, one scale for entire tensor
    "INT8 per-tensor": lambda t: quantize_symmetric(t, 8),
    # INT4 naive: highly sensitive to outliers (expect poor quality)
    "INT4 per-tensor": lambda t: quantize_symmetric(t, 4),
    # INT4 grouped: isolates outliers to their own group
    "INT4 per-group": lambda t: quantize_per_group(t, 4, 128),
    # NF4: information-optimal for Gaussian weights (bitsandbytes)
    "NF4": quantize_nf4,
}

# Run each method and collect error metrics
print(f"{"Method":<20} {"MSE":>10} {"SQNR (dB)":>10}")
print("-" * 42)
results = {}
for name, fn in methods.items():
    # Quantize weights and measure error vs original
    m = measure_error(weights, fn(weights))
    results[name] = m
    # Print: higher SQNR = better quality preservation
    print(f"{name:<20} {m["mse"]:>10.2e} {m["sqnr_db"]:>10.1f}")


In [ ]:
# === Bar chart comparing SQNR across methods ===
# Higher SQNR means less quantization noise relative to signal
fig, ax = plt.subplots(figsize=(9, 4))

# Extract results for plotting
names = list(results.keys())
sqnrs = [results[n]["sqnr_db"] for n in names]

# Color code by quality tier: blue=excellent, green=good, red=poor
colors = ["#3b82f6", "#10b981", "#ef4444", "#8b5cf6", "#f59e0b"]
ax.barh(names, sqnrs, color=colors)

# Reference line: 30 dB is the minimum acceptable quality threshold
ax.axvline(x=30, color="gray", ls="--", alpha=0.6, label="30 dB threshold")
ax.set_xlabel("SQNR (dB) — higher is better")
ax.set_title("Quantization Quality with Outlier-Heavy Weights")
ax.legend()
plt.tight_layout()
plt.show()
# Key insight: per-group INT4 dramatically beats per-tensor INT4


## 3. Memory Savings by Model Size

In [ ]:
# === Memory savings calculation across popular model sizes ===
# Compute weight memory at FP16, INT8, and INT4 for each model
# Demonstrates 4x savings from INT4 quantization
# Define popular model sizes for memory calculation
model_sizes_b = {"Llama-7B": 7, "Llama-13B": 13, "Llama-70B": 70, "Mixtral-8x7B": 46.7}

def mem_gb(params_b: float, bits: int) -> float:
    """Calculate memory in GB for a model at given bit width."""
    # bytes = params * bits / 8, convert to GB
    return params_b * 1e9 * bits / 8 / 1024**3

# Print memory table for each model at FP16/INT8/INT4
print(f"{"Model":<14} {"FP16 (GB)":>10} {"INT8 (GB)":>10} {"INT4 (GB)":>10} {"Savings":>8}")
print("-" * 56)
mem_data = []
for name, params in model_sizes_b.items():
    # Calculate memory at each precision level
    fp16 = mem_gb(params, 16)
    int8 = mem_gb(params, 8)
    int4 = mem_gb(params, 4)
    # Savings = reduction from FP16 to INT4
    savings = (1 - int4 / fp16) * 100
    mem_data.append({"name": name, "fp16": fp16, "int8": int8, "int4": int4})
    print(f"{name:<14} {fp16:>10.1f} {int8:>10.1f} {int4:>10.1f} {savings:>7.0f}%")

In [ ]:
# Grouped bar chart: memory per model at each precision
# Configure plot element
fig_6, ax_6 = plt.subplots(figsize=(10, 4))
x = np.arange(len(mem_data))
width = 0.25  # bar width for grouped layout

# Plot three bars per model (FP16, INT8, INT4)
# Iterate over each item
for i, (label, key, color) in enumerate([
    ("FP16", "fp16", "#3b82f6"),
    ("INT8", "int8", "#10b981"),
    ("INT4", "int4", "#f59e0b")]):
    vals = [d[key] for d in mem_data]
    ax_6.bar(x + i * width, vals, width, label=label, color=color)

# Label axes with model names
ax_6.set_xticks(x + width)
ax_6.set_xticklabels([d["name"] for d in mem_data])
ax_6.set_ylabel("Memory (GB)")
ax_6.set_title("Model Weight Memory by Quantization Level")
ax_6.legend()
# Configure plot element
plt.tight_layout()
# Configure plot element
plt.show()

## 4. Perplexity Impact Simulation

Simulate a language model head projection to measure how quantization degrades output quality.

In [ ]:
def compute_perplexity_delta(proj_weight: torch.Tensor, quant_fn) -> float:
    """Measure perplexity change from quantizing projection weights."""
    # Synthetic hidden states (batch of 256 tokens, dim 2048)
    torch.manual_seed(123)
    hidden = torch.randn(256, 2048) * 0.5
    # Reference logits from unquantized weights
    ref_logits = hidden @ proj_weight.T
    ref_probs = torch.softmax(ref_logits, dim=-1)
    # Quantized logits
    q_logits = hidden @ quant_fn(proj_weight).T
    # Cross-entropy measures quality degradation
    log_probs = torch.log_softmax(q_logits, dim=-1)
    ce = -(ref_probs * log_probs).sum(dim=-1).mean()
    # Return perplexity (lower is better)
    # Return the computed result
    return torch.exp(ce).item()

# Simulate vocab projection matrix (32K vocab x 2048 hidden)
torch.manual_seed(42)
proj_w = torch.randn(32000, 2048) * (2.0 / 2048) ** 0.5

# Measure perplexity for each quantization method
ppl_methods = {
    "FP32 (ref)": lambda t: t,
    "FP16": lambda t: t.to(torch.float16).to(torch.float32),
    "INT8": lambda t: quantize_symmetric(t, 8),
    "INT4 naive": lambda t: quantize_symmetric(t, 4),
    "INT4 group": lambda t: quantize_per_group(t, 4, 128),
    "NF4": quantize_nf4,
}

# Compute baseline perplexity first
base_ppl = compute_perplexity_delta(proj_w, lambda t: t)
# Display results to user
print(f"{"Method":<14} {"Perplexity":>12} {"Delta %":>10}")
# Display results to user
print("-" * 38)
ppl_results = {}
# Iterate over each item
for name, fn in ppl_methods.items():
    ppl = compute_perplexity_delta(proj_w, fn)
    delta_pct = (ppl - base_ppl) / base_ppl * 100
    ppl_results[name] = {"ppl": ppl, "delta_pct": delta_pct}
    # Display results to user
    print(f"{name:<14} {ppl:>12.2f} {delta_pct:>+9.2f}%")

## 5. GPU Fit Analysis

In [ ]:
# Check which models fit on which GPUs at each precision
gpus = {"A10G 24GB": 24, "RTX 4090 24GB": 24, "A100 80GB": 80, "H100 80GB": 80}

# Display results to user
print(f"{"Model":<14} {"Quant":<6} {"Size":>6} | ", end="")
# Display results to user
print(" | ".join(f"{g:<12}" for g in gpus.keys()))
# Display results to user
print("-" * 85)

# Iterate over each item
for name, params in model_sizes_b.items():
    # Iterate over each item
    for bits, qname in [(16, "FP16"), (8, "INT8"), (4, "INT4")]:
        size = mem_gb(params, bits)
        # Model fits if it uses <85% of GPU memory (leave room for KV cache)
        fits = ["  ✅" if size < v * 0.85 else "  ❌" for v in gpus.values()]
        # Display results to user
        print(f"{name:<14} {qname:<6} {size:>5.1f}G | {" | ".join(fits)}")
    print()  # blank line between models

## Key Takeaways

- **INT4 with AWQ/GPTQ**: <3% quality loss, 4x memory reduction, 2x decode speedup
- **Per-group quantization**: Essential for handling outliers; always use group_size=128
- **FP8 on H100**: Zero-config, near-lossless, native hardware support
- **KV cache quantization**: At high batch sizes, saves more memory than weight quantization
- **Method > bit-width**: AWQ-4bit beats naive-8bit. Always use a calibrated method.